In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import StringType, TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "mdt_ab_cost_ctr": MTU.get_fully_qualified_table_name("oracle_dev", "whsusr", "mdt_ab_cost_ctr"),
    "mdt_ab_zbb_cc_ent_assoc": MTU.get_fully_qualified_table_name("oracle_dev", "whsusr", "mdt_ab_zbb_cc_ent_assoc"),
    "mdt_ab_zbb_cost_ctr": MTU.get_fully_qualified_table_name("oracle_dev", "whsusr", "mdt_ab_zbb_cost_ctr"),
    "mdt_cost_ctr": MTU.get_fully_qualified_table_name("oracle_dev", "whsusr", "mdt_cost_ctr"),
    "pbcc_zbb_enty_dm": MTU.get_fully_qualified_table_name("oracle_dev", "whsusr", "pbcc_zbb_enty_dm"),
}

In [0]:
# create mdt_cost_ctr df
mdt_cost_ctr_df = (
    spark.read.table(SOURCE_TABLES["mdt_cost_ctr"])
    .alias("mdt_cost_cntr")
    .select(
        "cost_ctr_code",
        "cost_ctr_abbr",
        "cost_ctr_name",
        "budg_flg",
        "cost_ctr_post_lck_ind",
        "sap_upld_flg",
        "tam_flg",
        "zbb_flg",
        "cost_ctr_post_lck_ind",
    )
)

# Create Columns
mdt_cost_ctr_df = mdt_cost_ctr_df.withColumn("cntry_cd", F.lit("CAN")).withColumn("cmpy_cd", F.lit("0"))
mdt_cost_ctr_df = mdt_cost_ctr_df.withColumn(
    "sap_actv_flg", F.when(F.col("cost_ctr_post_lck_ind") == "1", "N").otherwise("Y")
)

# drop column
mdt_cost_ctr_df = mdt_cost_ctr_df.drop("cost_ctr_post_lck_ind")

# rename columns
mdt_cost_ctr_df = (
    mdt_cost_ctr_df.withColumnRenamed("cost_ctr_code", "cost_cntr_cd")
    .withColumnRenamed("cost_ctr_abbr", "cost_cntr_nm")
    .withColumnRenamed("cost_ctr_name", "cost_cntr_long_nm")
)

# add null columns
mdt_cost_ctr_df = (
    mdt_cost_ctr_df.withColumn("cmpy_grp_cd", F.lit(None).cast(StringType()))
    .withColumn("cmpy_nm", F.lit(None).cast(StringType()))
    .withColumn("bus_area_cd", F.lit(None).cast(StringType()))
    .withColumn("bus_area_nm", F.lit(None).cast(StringType()))
    .withColumn("ctrlling_area_cd", F.lit(None).cast(StringType()))
    .withColumn("enty_id", F.lit(None).cast(StringType()))
    .withColumn("l1_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l1_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l2_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l2_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l3_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l3_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l4_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l4_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l5_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l5_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l6_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l6_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("plng_tam_enty", F.lit(None).cast(StringType()))
    .withColumn("plng_budg_enty", F.lit(None).cast(StringType()))
    .withColumn("mkt_flg", F.lit(None).cast(StringType()))
    .withColumn("site_cd", F.lit(None).cast(StringType()))
    .withColumn("sub_fn_cd", F.lit(None).cast(StringType()))
    .withColumn("brnd_nm", F.lit(None).cast(StringType()))
    .withColumn("brnd_cd", F.lit(None).cast(StringType()))
    .withColumn("site_nm", F.lit(None).cast(StringType()))
    .withColumn("sub_fn_nm", F.lit(None).cast(StringType()))
)

In [0]:
# create mdt_ab_cost_ctr df
mdt_ab_cost_ctr_df = (
    spark.read.table(SOURCE_TABLES["mdt_ab_cost_ctr"])
    .alias("mdt_ab_cost_cntr")
    .select(
        "cost_ctr_code",
        "cost_ctr_abbr",
        "cost_ctr_name",
        "fin_comp_ref",
        "bus_area_ref",
        "ctrl_area_ref",
        "budg_flg",
        "cost_ctr_post_lck_ind",
        "sap_upld_flg",
        "tam_flg",
        "zbb_flg",
    )
)

# Create Columns
mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.withColumn("cntry_cd", F.lit("US")).withColumn("cmpy_cd", F.lit("0"))
mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.withColumn(
    "sap_actv_flg", F.when(F.col("cost_ctr_post_lck_ind") == "1", "N").otherwise("Y")
)
mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.withColumn(
    "cmpy_cd", F.coalesce(mdt_ab_cost_ctr_df["fin_comp_ref"], F.lit("0"))
)

# drop column
mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.drop("cost_ctr_post_lck_ind").drop("fin_comp_ref")

# rename columns
mdt_ab_cost_ctr_df = (
    mdt_ab_cost_ctr_df.withColumnRenamed("cost_ctr_code", "cost_cntr_cd")
    .withColumnRenamed("cost_ctr_abbr", "cost_cntr_nm")
    .withColumnRenamed("cost_ctr_name", "cost_cntr_long_nm")
    .withColumnRenamed("bus_area_ref", "bus_area_cd")
    .withColumnRenamed("ctrl_area_ref", "ctrlling_area_cd")
)

In [0]:
# create mdt_ab_zbb_cc_ent_assoc df
mdt_ab_zbb_cc_ent_assoc_df = (
    spark.read.table(SOURCE_TABLES["mdt_ab_zbb_cc_ent_assoc"])
    .alias("mdt_ab_zbb_cc_ent_assoc")
    .select("ab_zbb_cc_ent_assoc_id", "zbb_ent_ref")
)

# rename columns
mdt_ab_zbb_cc_ent_assoc_df = mdt_ab_zbb_cc_ent_assoc_df.withColumnRenamed(
    "ab_zbb_cc_ent_assoc_id", "cost_cntr_cd"
).withColumnRenamed("zbb_ent_ref", "zbb_enty_cd")

In [0]:
# create pbcc_zbb_enty_dm df
pbcc_zbb_enty_dm_df = (
    spark.read.table(SOURCE_TABLES["pbcc_zbb_enty_dm"])
    .alias("pbcc_zbb_enty_dm")
    .select(
        "zbb_enty_cd",
        "lvl_1_zbb_enty_cd",
        "lvl_1_zbb_enty_nm",
        "lvl_2_zbb_enty_cd",
        "lvl_2_zbb_enty_nm",
        "lvl_3_zbb_enty_cd",
        "lvl_3_zbb_enty_nm",
        "lvl_4_zbb_enty_cd",
        "lvl_4_zbb_enty_nm",
        "lvl_5_zbb_enty_cd",
        "lvl_5_zbb_enty_nm",
        "lvl_6_zbb_enty_cd",
        "lvl_6_zbb_enty_nm",
    )
)

# rename columns
pbcc_zbb_enty_dm_df = (
    pbcc_zbb_enty_dm_df.withColumnRenamed("cost_ctr_code", "cost_cntr_cd")
    .withColumnRenamed("cost_ctr_abbr", "cost_cntr_name")
    .withColumnRenamed("cost_ctr_name", "cost_cntr_long_nm")
    .withColumnRenamed("lvl_1_zbb_enty_cd", "l1_enty_cd")
    .withColumnRenamed("lvl_1_zbb_enty_nm", "l1_enty_nm")
    .withColumnRenamed("lvl_2_zbb_enty_cd", "l2_enty_cd")
    .withColumnRenamed("lvl_2_zbb_enty_nm", "l2_enty_nm")
    .withColumnRenamed("lvl_3_zbb_enty_cd", "l3_enty_cd")
    .withColumnRenamed("lvl_3_zbb_enty_nm", "l3_enty_nm")
    .withColumnRenamed("lvl_4_zbb_enty_cd", "l4_enty_cd")
    .withColumnRenamed("lvl_4_zbb_enty_nm", "l4_enty_nm")
    .withColumnRenamed("lvl_5_zbb_enty_cd", "l5_enty_cd")
    .withColumnRenamed("lvl_5_zbb_enty_nm", "l5_enty_nm")
    .withColumnRenamed("lvl_6_zbb_enty_cd", "l6_enty_cd")
    .withColumnRenamed("lvl_6_zbb_enty_nm", "l6_enty_nm")
)

In [0]:
# create mdt_ab_zbb_cost_cntr df
mdt_ab_zbb_cost_ctr_df = (
    spark.read.table(SOURCE_TABLES["mdt_ab_zbb_cost_ctr"])
    .alias("mdt_ab_zbb_cost_ctr")
    .select(
        "ab_zbb_cost_ctr_code", "ab_zbb_cost_ctr_name", "budg_flg", "sap_actv_flg", "sap_upld_flg", "tam_flg", "zbb_flg"
    )
)

# Create Columns
mdt_ab_zbb_cost_ctr_df = mdt_ab_zbb_cost_ctr_df.withColumn("cntry_cd", F.lit("US")).withColumn("cmpy_cd", F.lit("0"))
mdt_ab_zbb_cost_ctr_df = mdt_ab_zbb_cost_ctr_df.withColumn(
    "cost_cntr_long_nm", mdt_ab_zbb_cost_ctr_df["ab_zbb_cost_ctr_name"]
)

# rename columns
mdt_ab_zbb_cost_ctr_df = mdt_ab_zbb_cost_ctr_df.withColumnRenamed(
    "ab_zbb_cost_ctr_code", "cost_cntr_cd"
).withColumnRenamed("ab_zbb_cost_ctr_name", "cost_cntr_nm")

# add null columns
mdt_ab_zbb_cost_ctr_df = (
    mdt_ab_zbb_cost_ctr_df.withColumn("cmpy_grp_cd", F.lit(None).cast(StringType()))
    .withColumn("cmpy_nm", F.lit(None).cast(StringType()))
    .withColumn("bus_area_cd", F.lit(None).cast(StringType()))
    .withColumn("bus_area_nm", F.lit(None).cast(StringType()))
    .withColumn("ctrlling_area_cd", F.lit(None).cast(StringType()))
    .withColumn("enty_id", F.lit(None).cast(StringType()))
    .withColumn("l1_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l1_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l2_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l2_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l3_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l3_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l4_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l4_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l5_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l5_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("l6_enty_cd", F.lit(None).cast(StringType()))
    .withColumn("l6_enty_nm", F.lit(None).cast(StringType()))
    .withColumn("plng_tam_enty", F.lit(None).cast(StringType()))
    .withColumn("plng_budg_enty", F.lit(None).cast(StringType()))
    .withColumn("mkt_flg", F.lit(None).cast(StringType()))
    .withColumn("site_cd", F.lit(None).cast(StringType()))
    .withColumn("sub_fn_cd", F.lit(None).cast(StringType()))
    .withColumn("brnd_nm", F.lit(None).cast(StringType()))
    .withColumn("brnd_cd", F.lit(None).cast(StringType()))
    .withColumn("site_nm", F.lit(None).cast(StringType()))
    .withColumn("sub_fn_nm", F.lit(None).cast(StringType()))
)

In [0]:
try:
    # Perform joins on mdt_ab_cost_ctr_df
    mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.join(mdt_ab_zbb_cc_ent_assoc_df, ["cost_cntr_cd"], how="left")
    mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.join(pbcc_zbb_enty_dm_df, ["zbb_enty_cd"], how="left")

    # rename column
    mdt_ab_cost_ctr_df = mdt_ab_cost_ctr_df.withColumnRenamed("zbb_enty_cd", "enty_id")

    # add null columns
    mdt_ab_cost_ctr_df = (
        mdt_ab_cost_ctr_df.withColumn("cmpy_grp_cd", F.lit(None))
        .withColumn("cmpy_nm", F.lit(None))
        .withColumn("bus_area_nm", F.lit(None))
        .withColumn("plng_tam_enty", F.lit(None))
        .withColumn("plng_budg_enty", F.lit(None))
        .withColumn("mkt_flg", F.lit(None))
        .withColumn("site_cd", F.lit(None))
        .withColumn("sub_fn_cd", F.lit(None))
        .withColumn("brnd_nm", F.lit(None))
        .withColumn("brnd_cd", F.lit(None))
        .withColumn("site_nm", F.lit(None))
        .withColumn("sub_fn_nm", F.lit(None))
    )

except Exception as e:
    medallion.logger.error(f"Error unioning source tables. Error message: {e}")
    raise

In [0]:
try:
    # union tables
    medallion.df = mdt_ab_cost_ctr_df.unionByName(mdt_ab_zbb_cost_ctr_df)
    medallion.df = medallion.df.unionByName(mdt_cost_ctr_df).withColumn(
        "__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType())
    )

except Exception as e:
    medallion.logger.error(f"Error unioning source tables. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing company table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )